# Küresel Hava Kalitesi İndeksi (AQI) Analizi ve Makine Öğrenmesi ile Hava Kirliliği Sınıflandırma Sistemi
**Ders:** Araştırma Yöntem ve Teknikleri – Vize Projesi  
**Öğrenci:** Ayşegül Yılmaz (361526091)  
**GitHub:** https://github.com/ylmzaysegl

In [ ]:
# 1. Kütüphanelerin Yüklenmesi
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Kütüphaneler başarıyla yüklendi!')

In [ ]:
# 2. Veri Setinin Yüklenmesi (Colab sol menüden 'global air pollution dataset.csv' dosyasını yükleyiniz)
df = pd.read_csv('global air pollution dataset.csv')
print('Veri Seti Boyutu (Satır, Sütun):', df.shape)
display(df.head())

print('\nEksik Değer Sayıları:')
print(df.isnull().sum())

In [ ]:
# 3. Betimsel İstatistikler ve Keşifsel Veri Analizi (EDA)
num_cols = ['AQI Value', 'CO AQI Value', 'Ozone AQI Value', 'NO2 AQI Value', 'PM2.5 AQI Value']
display(df[num_cols].describe().round(2))

plt.figure(figsize=(9, 4.5))
order = df['AQI Category'].value_counts().index
sns.countplot(data=df, x='AQI Category', order=order, palette='viridis')
plt.title('AQI Kategorilerinin Veri Setindeki Dağılımı (n = 23.463)', fontsize=12, fontweight='bold')
plt.xlabel('AQI Kategorisi')
plt.ylabel('Gözlem Sayısı')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6.5, 5))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Kirletici Bileşenler ve AQI Değeri Korelasyon Matrisi', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 4. Veri Ön İşleme ve Eğitim/Test Ayrımı (%80 Eğitim - %20 Test)
X = df[['CO AQI Value', 'Ozone AQI Value', 'NO2 AQI Value', 'PM2.5 AQI Value']]
le = LabelEncoder()
y = le.fit_transform(df['AQI Category'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Eğitim Seti: {X_train.shape[0]} satır | Test Seti: {X_test.shape[0]} satır')

In [ ]:
# 5. Makine Öğrenmesi Modellerinin Eğitilmesi ve Karşılaştırılması
models = {
    'Gaussian Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f'=== {name} (Accuracy: {acc:.4f}) ===')
    print(classification_report(y_test, preds, target_names=le.classes_, zero_division=0))

# En iyi modellerden Random Forest için Confusion Matrix görselleştirme
rf_preds = models['Random Forest'].predict(X_test)
cm = confusion_matrix(y_test, rf_preds)
plt.figure(figsize=(7.5, 5.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Random Forest Karmaşıklık Matrisi (Confusion Matrix)', fontsize=12, fontweight='bold')
plt.xlabel('Tahmin Edilen Sınıf')
plt.ylabel('Gerçek Sınıf')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()